In [ ]:
# 强制 Jupyter 每次运行都重新加载被修改过的外部 .py 文件
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from core.dispatch_engine import EcoGridOptimizer

plt.style.use('dark_background')
sns.set_context("talk")

print("🔬 EcoGrid-Quant V3 敏感性分析试验田已就绪 (MACC / Grid Search)")


In [ ]:
# ==========================================
# 数据准备：复用 V1 真实慕尼黑气象数据 (本 notebook 独立自给自足)
# ==========================================
import requests

print("📡 正在连接 Open-Meteo 抓取慕尼黑真实小时级气象数据...")
api_url = (
    "https://archive-api.open-meteo.com/v1/archive?"
    "latitude=48.1371&longitude=11.5754&"
    "start_date=2024-05-01&end_date=2024-05-15&"
    "hourly=shortwave_radiation,wind_speed_100m&"
    "timezone=Europe%2FBerlin"
)
raw = requests.get(api_url).json()
df_munich = pd.DataFrame({
    "光伏辐射_W/m2": raw["hourly"]["shortwave_radiation"],
    "百米风速_m/s": raw["hourly"]["wind_speed_100m"],
})

# 物理换算口径与 V1 (01_energy_data Cell 8) 完全一致，保证横向可比
WIND_CAPACITY = 50.0    # 风电场最大装机 (MW)
SOLAR_CAPACITY = 50.0   # 光伏场最大装机 (MW)
HORIZON = 48            # 本次回测窗口：48 小时 (2 天)

df_window = df_munich.iloc[:HORIZON]
wind_real = np.clip(df_window["百米风速_m/s"].values * 3.0, 0, WIND_CAPACITY)
solar_real = np.clip(df_window["光伏辐射_W/m2"].values * 0.05, 0, SOLAR_CAPACITY)
demand_real = np.full(HORIZON, 120.0)  # 120MW 基础工业负荷

print(
    f"✅ 数据就绪：{HORIZON} 小时窗口 | "
    f"风电均值 {wind_real.mean():.1f}MW | 光伏均值 {solar_real.mean():.1f}MW"
)


In [ ]:
# ==========================================
# V3 终局：CAPEX vs OPEX 资本博弈与最优投资决策
# ==========================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# 1. 财务假设 (Financial Assumptions)
UNIT_CAPEX = 300000.0  # 电池单价：€300,000 / MWh
LIFESPAN_DAYS = 3650   # 寿命：10年
SIMULATION_DAYS = 2    # 本次回测：48小时 (2天)

target_tax = 150.0     # 锁定极端碳税场景：€150/吨
test_capacities = np.arange(0, 301, 20)  # 投资试探：从 0 到 300 MWh，步长 20

opex_48h_list = []
capex_48h_list = []
total_cost_list = []

print("💰 [Wall Street Quant] 正在核算资产摊销与最优容量...")

# 2. 燃烧算力：高频扫描投资规模
for cap in tqdm(test_capacities, desc="Scanning BESS Capacity"):
    # 计算当前容量下的 48小时摊销成本 (Amortized CAPEX)
    amortized_capex = (cap * UNIT_CAPEX / LIFESPAN_DAYS) * SIMULATION_DAYS
    capex_48h_list.append(amortized_capex)

    # 实例化引擎，计算该容量下能把 48小时运行成本压到多低 (OPEX)
    engine = EcoGridOptimizer(carbon_tax_rate=target_tax, bess_capacity=cap)
    res = engine.optimize_horizon(demand_real, wind_real, solar_real, horizon=48)

    opex = res.fun if res.success else np.nan
    opex_48h_list.append(opex)

    # 计算真实总持有成本 (Total Cost of Ownership, TCO)
    total_cost_list.append(amortized_capex + opex)

# 3. 寻找全场最低点 (最优投资决策)
min_cost_idx = np.argmin(total_cost_list)
optimal_capacity = test_capacities[min_cost_idx]
optimal_cost = total_cost_list[min_cost_idx]

# ==========================================
# 4. 研报级可视化：总持有成本 U 型曲线
# ==========================================
plt.figure(figsize=(12, 7))

# 绘制三条命运曲线
plt.plot(
    test_capacities, opex_48h_list, color='#00cec9', linewidth=2,
    linestyle='--', label='48H OPEX (Operating Cost + Carbon Tax)',
)
plt.plot(
    test_capacities, capex_48h_list, color='#e84393', linewidth=2,
    linestyle='-.', label='48H Amortized CAPEX (Battery Investment)',
)
plt.plot(
    test_capacities, total_cost_list, color='#fdcb6e', linewidth=4,
    label='Total Cost (OPEX + CAPEX)',
)

# 标出神圣拐点 (The Sweet Spot)
plt.scatter(optimal_capacity, optimal_cost, color='red', s=150, zorder=5)
plt.annotate(f'Optimal Investment:\n{optimal_capacity} MWh\nMin Cost: €{optimal_cost:,.0f}',
             xy=(optimal_capacity, optimal_cost),
             xytext=(optimal_capacity + 15, optimal_cost + 5000),
             arrowprops=dict(facecolor='white', arrowstyle='->'),
             color='white', fontsize=12, fontweight='bold')

plt.title(
    f"V3: Capital Allocation under €{target_tax}/t Carbon Risk",
    fontsize=18, fontweight='bold', color='white', pad=20,
)
plt.xlabel("Battery Energy Storage Capacity (MWh)", fontsize=14, color='lightgrey')
plt.ylabel("48-Hour Equivalent Cost (€)", fontsize=14, color='lightgrey')
plt.legend(loc='upper right', frameon=True, facecolor='black', edgecolor='white')
plt.grid(True, linestyle=':', alpha=0.3)
plt.tight_layout()
plt.show()
